# 9장 - 음성을 텍스트로 (STT)

원본 파일: `chap09/stt_basic.py`

In [3]:
"""음성을 텍스트로 변환하기 (STT) - OpenRouter 오디오 채팅 모델 사용

OpenRouter는 Whisper 전용 엔드포인트(client.audio.transcriptions)를 지원하지 않지만,
오디오를 입력으로 받는 멀티모달 채팅 모델(gemini-2.5-flash 등)에 오디오를 넣고
"받아써줘"라고 요청하면 STT가 된다. 로컬 모델·별도 API 키가 필요 없다.
"""
from openai import OpenAI
from dotenv import load_dotenv
import os
import base64

load_dotenv()

BASE_DIR = (os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd())

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENAI_API_KEY"),
)

# 오디오 입력을 지원하는 멀티모달 모델 (받아쓰기 품질이 로컬 whisper-tiny보다 훨씬 좋음)
AUDIO_MODEL = "google/gemini-2.5-flash"

In [4]:
def audio_to_text(audio_file_path: str, instruction: str) -> str:
    """오디오 파일을 채팅 모델에 넣어 지시(instruction)대로 텍스트를 받아온다."""
    with open(audio_file_path, "rb") as f:
        audio_b64 = base64.b64encode(f.read()).decode()

    audio_format = os.path.splitext(audio_file_path)[1].lstrip(".")  # 'mp3', 'wav' 등

    response = client.chat.completions.create(
        model=AUDIO_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "input_audio", "input_audio": {"data": audio_b64, "format": audio_format}},
            ],
        }],
    )
    return response.choices[0].message.content

## 실행 / 테스트

In [5]:
audio_path = os.path.join(BASE_DIR, "data", "lsy_audio_2023_58s.mp3")

In [6]:
print("===== 1. 받아쓰기 (한국어 그대로) =====")
print(audio_to_text(audio_path, "이 오디오 내용을 한국어로 그대로 받아써줘. 받아쓴 내용만 출력."))

===== 1. 받아쓰기 (한국어 그대로) =====
안녕하세요. 어 이 강의는 GPT API로 챗봇 만들기라는 내용을 다루는 강의입니다. 어 GPT API에 대해서 뭐 생소하신 분들도 있을 텐데 어 우리가 잘 알고 있는 챗 GPT. 챗 GPT를 이용해 챗 GPT 기능을 이용해서 어 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 어 이야기할 거예요. 그래서 뭐 이런 강의들이 사실 많이 있습니다. 그래서 뭐 여러 가지들이 있는데 좀 이 강의의 특징이라고 한다면 이제 GPT로 명확한 미션을 달성하는 챗봇 프로그램은 만드는 게 사실 쉽지는 않은데 이거를 어떻게 해서 구현을 하는지 그리고 그 왜 필요한지에 대해서 좀 이야기를 할 거고요. 어 그 예제로 뭐 예제 여러 가지가 될 수 있는데 여기에서 예제로 하는 거는 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램을 만드는 거를 다루려고 합니다. 네, 그래서 프로그램이 실행되는 모습을 한번 보여 드릴게요. 그 우리가 그 만들 프로그램은 이런 식으로 이제 나타나게 되고요.


In [7]:
print("\n===== 2. 영어로 번역 =====")
print(audio_to_text(audio_path, "이 오디오 내용을 영어로 번역해서 받아써줘. 번역한 내용만 출력."))


===== 2. 영어로 번역 =====
Hello. This lecture is about 'How to build a chatbot using the GPT API'. Some of you might be unfamiliar with the GPT API, but we'll be talking about how to create the program we want using the features of ChatGPT, which we are all familiar with. There are many such lectures out there, but a distinctive feature of this lecture is that it focuses on building a chatbot program that achieves a clear mission with GPT. It's not easy to implement this, so we'll talk about how to implement it and why it's necessary. As an example, there could be several, but the example we will use here is creating a program that automatically generates music playlist videos through conversation. I'll show you how the program runs. The program we're going to create will appear like this.
